# Chapter 3 — Provenance: span ↔ triple

Every fact a GEODE-RAG answer rests on points back to the exact source
span it came from — `file:line:char`, plus the raw substring. This
notebook reconstructs that span↔triple map for the Northwind FY2025
annual report with `ProvenanceLedger`, shows a prose triple resolving as
`unaligned`, and proves a resolved span actually supports its triple with
`is_consistent`.

We do **not** load the trained store or Qwen here; provenance alignment is
pure text→span work over the markdown source.

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get("GMS_RAG_TUTORIAL", "/home/asudjianto/jupyterlab/agent-tutorial-private/beyond-chunk-and-pray/code"))
from forgeloop.rag import load_store, CORPUS, STORE, EVAL_COHORT, DEVICE, store_config

In [ ]:
from knowlytix.knowledge.geode.provenance import (
    ProvenanceLedger,
    Provenance,
    is_consistent,
    canon,
)

REPORT = CORPUS
ledger = ProvenanceLedger(REPORT)
print("ledger built over", REPORT)

## Listing 1 — resolve a segment-revenue triple to its table cell

The triple `(cloud platform, has_revenue, 120.0)` was extracted from a table
cell. `resolve` re-aligns it post-hoc and returns a `Provenance`: the line,
the absolute char span, the raw cell text, and the alignment `method`.

In [ ]:
p = ledger.resolve("cloud platform", "has_revenue", "120.0")
print("location:", p.location())
print("method:  ", p.method)
print("raw:     ", repr(p.raw))
print("line:    ", repr(ledger.line(p.line_no)))

Expected output (grounded in the corpus, line 15 = the Cloud Platform row):

```
location: data/annual_report.md:15:553-558
method:   table_cell
raw:      '120.0'
line:     '| Cloud Platform | Technology | 120.0 | 340 |'
```

The `method` is `table_cell`: the figure is anchored to a precise span, not
parsed loose from prose. The `raw` substring is the cell text itself — byte‑
exact `120.0`, the same value the store's ENM returns (Ch.~\ref{ch:store}).

## Listing 2 — a prose triple is `unaligned`

The MD&A section says the Cloud Platform segment “continued to lead the
company's topline.” That sentence carries no table cell, schema bullet, or
section header for a `(cloud platform, leads, topline)` triple. Post-hoc
structural alignment cannot find an anchor, so it resolves as `unaligned`.

In [ ]:
prose = ledger.resolve("cloud platform", "leads", "topline")
print("location:", prose.location())
print("method:  ", prose.method)
print("raw:     ", repr(prose.raw))
print("aligned? ", prose.method != "unaligned")

Expected output:

```
location: data/annual_report.md:-1:-1--1
method:   unaligned
raw:      ''
aligned?  False
```

`line_no` and the char offsets are `-1`: the sentinel for *no structural
anchor*. This is the honest signal that prose triples need spans captured at
**extraction** time — `ProvenanceLedger` re-aligns regex-extracted structured
triples (tables, schema bullets, headers) only. It does not invent a span for
an LLM-extracted prose claim.

## Listing 3 — the provenance-span trick: coarse triples cover prose

A prose section *can* still get a span — not for a fine-grained fact, but at
section granularity via an `in_section` triple keyed on the section slug. The
ledger resolves it to the **section header** line, giving the answer layer a
bounded region to cite even where no table fact exists.

In [ ]:
# section slugs are derived from headers: '## 6. Management Discussion and
# Analysis' -> 'management_discussion_and_analysis'
sec = ledger.resolve("md&a note", "in_section",
                     "management_discussion_and_analysis")
print("location:", sec.location())
print("method:  ", sec.method)
print("raw:     ", repr(sec.raw))

Expected output (the MD&A header is on line 60):

```
location: data/annual_report.md:60:1527-1567
method:   section_header
raw:      '## 6. Management Discussion and Analysis'
```

This is the *provenance-span trick*: a coarse `in_section` triple anchors a
prose region to its header span. It does not make the prose ENM-checkable —
it only gives a citable location. Fine numeric facts still come from tables
(Listing 1).

## Listing 4 — verify a resolved span actually supports its triple

Resolving to a span is not enough; the span must *support* the triple.
`is_consistent` re-reads the raw substring and checks it against the tail —
numeric-aware for table cells, slug-aware for headers and schema bullets.

In [ ]:
good = ledger.resolve("total", "has_revenue", "355.0")
print("total revenue:", good.location(), "raw=", repr(good.raw),
      "consistent=", is_consistent(good))

# An unaligned prose triple can never be consistent.
print("prose triple:  consistent=", is_consistent(prose))

Expected output:

```
total revenue: data/annual_report.md:19:698-703 raw= '355.0' consistent= True
prose triple:  consistent= False
```

`is_consistent` parses `355.0` out of the raw cell and matches it to the tail
within `1e-6`. The unaligned prose triple has no span to read, so it is
correctly `False` — the system refuses to claim support it cannot point to.

## Exercise (worked) — build a ledger over many triples and audit consistency

Resolve a batch of segment triples with `ledger.build(...)` and confirm every
aligned one is consistent.

In [ ]:
triples = [
    ("cloud platform", "has_revenue", "120.0"),
    ("devices", "has_revenue", "80.0"),
    ("logistics", "has_revenue", "95.0"),
    ("retail", "has_revenue", "60.0"),
    ("total", "has_revenue", "355.0"),
]
provs = ledger.build(triples)
for key, pr in provs.items():
    flag = "OK " if is_consistent(pr) else "BAD"
    print(flag, key, "->", pr.location(), repr(pr.raw))

Expected: every row prints `OK` — each segment-revenue triple aligns to a
table cell whose raw value equals the tail. The Total row aligns too, so the
sum anchor (Total = Σ segments, Ch.~\ref{ch:geode}) has a provenance to cite
if it ever breaks.

## Self-check — a resolved triple's `raw` equals the table value

The chapter's claim: a resolved segment-revenue triple carries the exact
table cell as its provenance, and the cell supports the triple.

In [ ]:
p = ledger.resolve("cloud platform", "has_revenue", "120.0")

# 1. the raw span IS the table value, byte-exact
assert p.raw == "120.0", p.raw
# 2. it aligned to a table cell with a real char span
assert p.method == "table_cell"
assert p.line_no == 15 and p.char_start >= 0 and p.char_end > p.char_start
# 3. the span actually supports the triple
assert is_consistent(p)
# 4. a prose triple with no structural anchor is unaligned + inconsistent
u = ledger.resolve("cloud platform", "leads", "topline")
assert u.method == "unaligned" and u.line_no == -1
assert not is_consistent(u)
print("OK: span<->triple provenance verified; prose triple correctly unaligned")